# Music Library Report

Paste this notebook in the root directory of your music collection and run the cells in order.
It scans every subdirectory for audio files, summarizes them in a structured table, and highlights duplicates.

## 1. Imports and configuration
Update the path below only if you want to analyze a folder other than the one that contains the notebook.
You can also tweak the list of extensions to match your collection.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import hashlib

try:
    import pandas as pd
except ImportError:
    pd = None

LIBRARY_ROOT = Path('.').resolve()  # Change this if needed.
AUDIO_EXTENSIONS = {
    '.aac', '.aiff', '.flac', '.m4a', '.m4b', '.mp3', '.oga', '.ogg',
    '.opus', '.wav', '.wma'
}

## 2. Helper functions
These utilities collect file metadata, compute hashes for duplicate detection, and build the requested reports.

In [ ]:
def scan_music_library(root_path: Path, extensions) -> list:
    """Return a list of dictionaries describing every music file under root_path."""
    root_path = root_path.resolve()
    records = []
    for path in root_path.rglob('*'):
        if path.is_file() and path.suffix.lower() in extensions:
            stats = path.stat()
            records.append({
                'relative_path': str(path.relative_to(root_path)),
                'file_name': path.name,
                'stem': path.stem,
                'extension': path.suffix.lower(),
                'size_bytes': stats.st_size,
                'modified': datetime.fromtimestamp(stats.st_mtime).isoformat(timespec='seconds'),
                'full_path': path
            })
    return records


def compute_file_hash(path: Path, chunk_size: int = 1_048_576) -> str:
    """Compute an MD5 hash for the file; chunked to avoid memory spikes."""
    hasher = hashlib.md5()
    with path.open('rb') as handle:
        while True:
            data = handle.read(chunk_size)
            if not data:
                break
            hasher.update(data)
    return hasher.hexdigest()


def find_duplicate_files(records: list) -> list:
    hash_map = defaultdict(list)
    for record in records:
        file_hash = compute_file_hash(record['full_path'])
        hash_map[file_hash].append(record)
    duplicates = [group for group in hash_map.values() if len(group) > 1]
    return duplicates


def find_multi_extension_groups(records: list) -> list:
    groups = defaultdict(list)
    for record in records:
        relative = Path(record['relative_path'])
        parent = str(relative.parent) or '.'
        key = (parent, record['stem'])
        groups[key].append(record)
    multi_ext_groups = []
    for (parent, stem), group in groups.items():
        extensions = {item['extension'] for item in group}
        if len(extensions) > 1:
            multi_ext_groups.append({
                'folder': parent,
                'stem': stem,
                'files': group
            })
    return multi_ext_groups

## 3. Scan the library
Run the next cell to build the master list of music files.

In [ ]:
music_files = scan_music_library(LIBRARY_ROOT, AUDIO_EXTENSIONS)
print(f"Scanned {len(music_files)} music files in {LIBRARY_ROOT}")

## 4. Structured list of music files
Displays a sortable table when pandas is available; otherwise prints a concise summary.

In [ ]:
display_records = [{k: v for k, v in record.items() if k != 'full_path'} for record in music_files]
sorted_records = sorted(display_records, key=lambda item: item['relative_path'])
if pd:
    display(pd.DataFrame(sorted_records))
else:
    for record in sorted_records:
        print(record)

## 5. Exact duplicate files (same content)
Files listed together share identical hashes. Review before deleting to ensure they are true duplicates.

In [ ]:
        duplicate_groups = find_duplicate_files(music_files)
        if not duplicate_groups:
            print('No byte-for-byte duplicate files found.')
        else:
            for idx, group in enumerate(duplicate_groups, start=1):
                print(f"
Duplicate group #{idx}:")
                for record in group:
                    print(f"  - {record['relative_path']} ({record['size_bytes']} bytes)")

## 6. Same recording exported to multiple formats
Detects files that share the same name within a folder but use different extensions (e.g., `song.mp3` and `song.wav`).

In [ ]:
        multi_ext_groups = find_multi_extension_groups(music_files)
        if not multi_ext_groups:
            print('No same-name files with different extensions were found in the same folder.')
        else:
            for group in multi_ext_groups:
                print(f"
Folder: {group['folder']} — Track name: {group['stem']}")
                for record in sorted(group['files'], key=lambda item: item['extension']):
                    print(f"  - {record['relative_path']} ({record['extension']})")